# Robustness leaderboard — tight-chord vs AdamW across datasets and a cross-model cell

Pilot scope (per `~/.claude/plans/as-part-of-our-tender-quilt.md`): does tight-chord's eval-loss advantage at the Phase L horizon transfer when we vary the dataset or the base model?

**Slate** (all at OLMo-2-1B × seq=2048 × global_batch=16 × packed_v1.1 × 9000 steps × constant LR × α=r × all-linear × bf16 × compile × single-GPU Blackwell unless noted):

| Cell | Base | Dataset | Rank | LRs per opt |
|---|---|---|---|---|
| `tulu3_r64` | OLMo-2-1B | Tulu-3 SFT mixture (400k, chat-templated) | 64 | AdamW {3e-5, 1e-4, 3e-4}; tight {3e-3, 1e-2, 3e-2} |
| `tulu3_r256` | OLMo-2-1B | Tulu-3 SFT mixture (400k) | 256 | same |
| `metamath_r64` | OLMo-2-1B | MetaMathQA (395k) | 64 | same |
| `metamath_r256` | OLMo-2-1B | MetaMathQA (395k) | 256 | same |
| `llama32_opc_r64` | Llama-3.2-1B | opc-sft-stage2 (436k) | 64 | same |

Pilot: seed=0, no rerun. Direction at any of the 3 LRs is the binary pass/fail.

**σ anchor**: no per-dataset multi-seed AdamW run yet. We quote Δ in `σ_AdamW(packed_v1, opc-sft-stage2, r-matched)` as a **proxy only** (r=64 → 0.0017, r=256 → 0.0017 same proxy). σ-unit Δ values must be re-anchored per (dataset, rank) before paper writeup.

**Pass criterion** (per plan): ≥5/6 dataset-axis cells with Δ < 0 and |Δ| > 1× ported-σ ⇒ promote to Phase-L-analog. Direction-only (Δ < 0) for the cross-model cell.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from lora_playground.loader import load_runs
from lora_playground.plotting import compare_variants_figure
from lora_playground.plotting.colors import OPTIM_COLORS, OPTIM_MARKERS

OPT_ADAMW = 'adamw'
OPT_CT    = 'adam-polar-product-lora-coupled-spectral-chord-tight'

# (cell_label, AdamW group, tight-chord group, lora_r, dataset/model description, sigma_ref proxy)
CELLS = [
    ('tulu3_r64',       'adamw_robustness_tulu3_1b_r64_blackwell',       'chord_tight_robustness_tulu3_1b_r64_blackwell',       64,  'OLMo-2-1B × Tulu-3',     0.0017),
    ('tulu3_r256',      'adamw_robustness_tulu3_1b_r256_blackwell',      'chord_tight_robustness_tulu3_1b_r256_blackwell',      256, 'OLMo-2-1B × Tulu-3',     0.0017),
    ('metamath_r64',    'adamw_robustness_metamath_1b_r64_blackwell',    'chord_tight_robustness_metamath_1b_r64_blackwell',    64,  'OLMo-2-1B × MetaMathQA', 0.0017),
    ('metamath_r256',   'adamw_robustness_metamath_1b_r256_blackwell',   'chord_tight_robustness_metamath_1b_r256_blackwell',   256, 'OLMo-2-1B × MetaMathQA', 0.0017),
    ('llama32_opc_r64', 'adamw_robustness_llama32_1b_opc_r64_blackwell', 'chord_tight_robustness_llama32_1b_opc_r64_blackwell', 64,  'Llama-3.2-1B × opc',     0.0017),
]

ALL_GROUPS = sorted({g for _, ga, gt, *_ in CELLS for g in (ga, gt)})
print(f'{len(CELLS)} cells × 2 groups each = {len(ALL_GROUPS)} log groups')

def variant_key(cfg):
    opt = cfg.get('optimizer')
    if opt == OPT_ADAMW:    return 'AdamW'
    if opt == OPT_CT:       return 'chord-tight k=1'
    return None

VARIANT_COLORS  = {'AdamW': OPTIM_COLORS.get(OPT_ADAMW, 'black'),
                   'chord-tight k=1': OPTIM_COLORS.get(OPT_CT, 'tab:red')}
VARIANT_MARKERS = {'AdamW': OPTIM_MARKERS.get(OPT_ADAMW, 'o'),
                   'chord-tight k=1': OPTIM_MARKERS.get(OPT_CT, 's')}

## Leaderboard summary — Δ(tight − AdamW) per cell

Single-table view across all 5 cells. Direction (Δ < 0) and σ-unit magnitude are the pass/fail signals. Best-LR-per-cell is computed by `compare_variants_figure` internals (argmin over LR grid).

In [ ]:
rows = []
per_cell_summaries = {}

for label, g_adamw, g_tight, r, desc, sigma in CELLS:
    groups = [g_adamw, g_tight]
    runs = load_runs(where={'log_group': groups}, logs_root='../logs', warn_cross_commit=False, quiet=True)
    if not runs:
        rows.append({'cell': label, 'desc': desc, 'r': r, 'adamw_best_lr': None, 'adamw_final': None,
                     'tight_best_lr': None, 'tight_final': None, 'delta': None, 'delta_sigma': None, 'n_runs': 0})
        continue
    # Dedup: keep longest trajectory per (optimizer, lr).
    dedup = {}
    for cfg, hist in runs:
        k = (cfg['optimizer'], float(cfg['lr']))
        if k not in dedup or len(hist) > len(dedup[k][1]):
            dedup[k] = (cfg, hist)
    runs_dd = list(dedup.values())
    fig, table_df, summary_df = compare_variants_figure(
        variants={'AdamW': {'optimizer': OPT_ADAMW}, 'chord-tight k=1': {'optimizer': OPT_CT}},
        common_where={'lora_r': r},
        ref_label='AdamW',
        sigma_ref=sigma,
        max_steps=9000,
        allow_partial=True,
        prefetched_runs=runs_dd,
        variant_key=variant_key,
        colors=VARIANT_COLORS, markers=VARIANT_MARKERS,
        suptitle=f'{label}: {desc} (r={r})',
        figsize=(11, 4),
    )
    plt.close(fig)  # rendered in per-cell figures below
    per_cell_summaries[label] = (table_df, summary_df, fig)
    adamw_row = summary_df[summary_df['variant'] == 'AdamW'].iloc[0] if 'AdamW' in summary_df['variant'].values else None
    tight_row = summary_df[summary_df['variant'] == 'chord-tight k=1'].iloc[0] if 'chord-tight k=1' in summary_df['variant'].values else None
    rows.append({
        'cell': label, 'desc': desc, 'r': r,
        'adamw_best_lr':  adamw_row['best_lr']  if adamw_row is not None else None,
        'adamw_final':    adamw_row['final']    if adamw_row is not None else None,
        'tight_best_lr':  tight_row['best_lr']  if tight_row is not None else None,
        'tight_final':    tight_row['final']    if tight_row is not None else None,
        'delta':          tight_row['delta']        if tight_row is not None else None,
        'delta_sigma':    tight_row['delta_sigma']  if tight_row is not None else None,
        'n_runs': len(runs_dd),
    })

leaderboard = pd.DataFrame(rows)
display(leaderboard.style.format({
    'adamw_best_lr': '{:.1e}', 'tight_best_lr': '{:.1e}',
    'adamw_final':   '{:.4f}', 'tight_final':   '{:.4f}',
    'delta':         '{:+.4f}','delta_sigma':   '{:+.2f}σ',
}, na_rep='—'))

# Pass criterion summary
dataset_axis = leaderboard[leaderboard['cell'].str.startswith(('tulu3', 'metamath'))]
n_pass = ((dataset_axis['delta'] < 0) & (dataset_axis['delta_sigma'].abs() > 1)).sum()
print(f'\nDataset-axis cells passing (Δ<0 AND |Δ|>1σ): {n_pass}/{len(dataset_axis)}')
cross_model = leaderboard[leaderboard['cell'] == 'llama32_opc_r64']
if len(cross_model):
    direction = '✓' if cross_model.iloc[0]['delta'] is not None and cross_model.iloc[0]['delta'] < 0 else '✗'
    print(f'Cross-model (Llama-3.2-1B × opc): direction {direction}')

## Per-cell figures

One `compare_variants_figure` panel per cell: left = final eval_loss vs η (log-x); right = best-η trajectory. Use these to diagnose LR-pinning at grid boundaries (if any AdamW or tight best LR sits at the edge, the grid needs extension before claiming a result).

In [ ]:
for label, g_adamw, g_tight, r, desc, sigma in CELLS:
    if label not in per_cell_summaries:
        print(f'{label}: no runs loaded — skipping')
        continue
    table_df, summary_df, _ = per_cell_summaries[label]
    # Re-call to render fresh figure (the cached fig was closed for layout reasons).
    groups = [g_adamw, g_tight]
    runs = load_runs(where={'log_group': groups}, logs_root='../logs', warn_cross_commit=False, quiet=True)
    dedup = {}
    for cfg, hist in runs:
        k = (cfg['optimizer'], float(cfg['lr']))
        if k not in dedup or len(hist) > len(dedup[k][1]):
            dedup[k] = (cfg, hist)
    fig, _t, _s = compare_variants_figure(
        variants={'AdamW': {'optimizer': OPT_ADAMW}, 'chord-tight k=1': {'optimizer': OPT_CT}},
        common_where={'lora_r': r},
        ref_label='AdamW', sigma_ref=sigma, max_steps=9000, allow_partial=True,
        prefetched_runs=list(dedup.values()), variant_key=variant_key,
        colors=VARIANT_COLORS, markers=VARIANT_MARKERS,
        suptitle=f'{label}: {desc} (r={r})',
        figsize=(11, 4),
    )
    plt.show()
    print(f'--- {label} per-η table ---')
    display(table_df.style.format('{:.4f}', na_rep='—'))
    print(f'--- {label} summary ---')
    display(summary_df.style.format({
        'best_lr': '{:.1e}', 'final': '{:.4f}', 'delta': '{:+.4f}', 'delta_sigma': '{:+.2f}σ'
    }, na_rep='—'))

## Notes for promotion to full Phase-L analog

If a dataset-axis cell passes (Δ<0, |Δ|>1σ_proxy):
- Add a 4-seed AdamW multi-seed run on that dataset to anchor the real σ.
- Re-quote Δ against the new σ before any paper-writeup claim.
- Add seed=1 rerun for both opts at best-LR (pass: |Δ(seed=0) − Δ(seed=1)| ≤ 2σ).

If a cell pins at the LR-grid boundary (AdamW best at 3e-5 or 3e-4; tight best at 3e-3 or 3e-2): extend the grid in that direction by half a decade before claiming the cell.